In [1]:
!pip install optuna

In [ ]:
!pip install lightgbm

#### Importing required libraries 

In [25]:
from utils import Load_Rumours_Dataset_filtering_since_first_post
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split,StratifiedKFold
from sklearn.metrics import *
import pandas as pd
import time
import optuna
from lightgbm import LGBMClassifier
import warnings
warnings.filterwarnings("ignore")

In [26]:
file_path_replies = r"../replies_sydneysiege.pkl"
file_path_posts = r"../posts_sydneysiege.pkl"

#### Testing a single load 

In [3]:
processor = Load_Rumours_Dataset_filtering_since_first_post(file_path_replies, file_path_posts, time_cut=1500)
processor.load_data()
processor.process_data()
train,test= processor.get_final_dataframes()


In [5]:
train.head()

,followers,favorite_count,retweet_count,first_time_diff,replies,no_verified,verified,embeddings_avg,rumour,min_since_fst_post
0,-0.201982,-0.244681,0.448889,3.018826,-0.1,1,0,"[0.029599157014959736, -0.1258555807565388, 0....",1,74.67
1,8.459877,-0.287234,-0.071111,1.377630,0.4,1,0,"[0.19377462948775953, 0.10071799257356259, 0.1...",1,77.02
2,-0.206237,0.755319,1.306667,0.232558,-0.1,1,0,"[0.1510988038033247, 0.2517209962010384, -0.08...",1,146.07
3,1.112561,-0.329787,-0.204444,0.017719,-0.5,0,1,"[0.11098087765276432, 0.10974890080979094, 0.0...",1,177.13
4,1.112403,3.734043,6.946667,-0.352159,2.5,0,1,"[0.1422940082848072, 0.13133273070508783, 0.15...",1,230.95


In [6]:
previous_node_count = 0

In [11]:
X_train  = train.drop(columns=['rumour'])
X_train = np.hstack([X_train.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_train.embeddings_avg.tolist()))])
#X = np.hstack([X.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X.embeddings_avg.tolist()))])
y_train =train['rumour']

X_test  = test.drop(columns=['rumour'])
X_test_new = test.iloc[previous_node_count:].drop(columns=['rumour'])

X_test_new =  np.hstack([X_test_new.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_test_new.embeddings_avg.tolist()))])
X_test = np.hstack([X_test.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_test.embeddings_avg.tolist()))])


y_test =test['rumour']
y_test_new = test.iloc[previous_node_count:]['rumour']

previous_node_count = test.shape[0]
print(f"New Instances: {X_test_new.shape[0]}")

New Instances: 303


#### Tunning Light GBM

In [14]:
previous_node_count = 0

In [15]:
### Tuning with all data
processor = Load_Rumours_Dataset_filtering_since_first_post(file_path_replies, file_path_posts, time_cut=3*24*60)
processor.load_data()
processor.process_data()
train,test= processor.get_final_dataframes()


In [16]:
X_train  = train.drop(columns=['rumour'])
X_train = np.hstack([X_train.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_train.embeddings_avg.tolist()))])
#X = np.hstack([X.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X.embeddings_avg.tolist()))])
y_train =train['rumour']

X_test  = test.drop(columns=['rumour'])
X_test_new = test.iloc[previous_node_count:].drop(columns=['rumour'])

X_test_new =  np.hstack([X_test_new.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_test_new.embeddings_avg.tolist()))])
X_test = np.hstack([X_test.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_test.embeddings_avg.tolist()))])


y_test =test['rumour']
y_test_new = test.iloc[previous_node_count:]['rumour']

previous_node_count = test.shape[0]
print(f"New Instances: {X_test_new.shape[0]}")

New Instances: 303


In [ ]:

n_train = len(X_train)

def objective(trial, X, y):
    param_grid = {
    # number of trees: keep reasonable upper bound for small dataset
    "n_estimators": trial.suggest_int("n_estimators", 50, 300, step=25),

    # learning rate: similar but slightly wider
    "learning_rate": trial.suggest_loguniform("learning_rate", 1e-4, 1e-2),

    # tree complexity: allow more variety but avoid huge trees for tiny data
    "num_leaves": trial.suggest_int("num_leaves", 3, 64, step=1),
    "max_depth": trial.suggest_int("max_depth", 2, 5),

    # min data in leaf: relative to training size (never larger than n_train)
    # lower bound 1, upper bound floor(n_train * 0.2) ensures splits are possible
    "min_data_in_leaf": trial.suggest_int(
        "min_data_in_leaf",
        1,
        max(2, int(max(2, n_train * 0.2))),
    ),

    # regularization: keep wide but avoid 0 lower bound
    "lambda_l1": trial.suggest_loguniform("lambda_l1", 1e-4, 100.0),
    "lambda_l2": trial.suggest_loguniform("lambda_l2", 1e-4, 100.0),

    # subsampling / feature fraction: use continuous suggestions
    "bagging_fraction": trial.suggest_uniform("bagging_fraction", 0.4, 1.0),
    # bagging frequency: allow 0 (no bagging) up to small integers
    "bagging_freq": trial.suggest_int("bagging_freq", 0, 10),
    "feature_fraction": trial.suggest_uniform("feature_fraction", 0.4, 1.0),
    }
    
    cv = StratifiedKFold(n_splits = 4, shuffle = True, random_state = 1337)
    cv_scores = np.empty(4)
    
    for idx, (train_idx, test_idx) in enumerate(cv.split(X, y)):
        X_train_fold, X_test_fold = X.iloc[train_idx], X.iloc[test_idx]
        y_train_fold, y_test_fold = y[train_idx], y[test_idx]

        model = LGBMClassifier(objective="binary", **param_grid,verbosity=-1,
                            seed= 1337,
                            feature_fraction_seed= 1337,
                            bagging_seed= 1337,
                            drop_seed= 1337,
                            data_random_seed= 1337
                            #class_weight= {0: neg_class_weight, 1: 1.0}
                              )
        #puning_callback = optuna.integration.LightGBMPruningCallback(trial, "auc")
        
        model.fit(
                X_train_fold,
                y_train_fold,
                eval_set = [(X_test_fold, y_test_fold)],
                eval_metric="auc")

        
        
        y_train_prob = model.predict_proba(X_train_fold)[:, 1]
        thresholds = np.linspace(0.01, 0.99, 100)
        f1_scores = [f1_score(y_train_fold, (y_train_prob > t).astype(int)) for t in thresholds]
        best_idx = np.argmax(f1_scores)
        best_threshold = thresholds[best_idx]


        y_train_pred = (y_train_prob > best_threshold).astype(int)
        f1_score_idx = f1_score(y_train_fold, y_train_pred)
       

        # Report and prune
        trial.report(f1_score_idx, step=idx)
        if trial.should_prune():
            raise optuna.TrialPruned()
        
        ### get F-beta score on top 

        cv_scores[idx] = f1_score_idx

    return np.mean(cv_scores)




In [ ]:
# Run study
start_time = time.time()

study = optuna.create_study(direction="maximize", study_name="LGBM Charlie Hebdo",
                            sampler=optuna.samplers.TPESampler(seed=111113857),
                            pruner=optuna.pruners.MedianPruner())
func = lambda trial: objective(trial, pd.DataFrame(X_train), y_train.astype(int))
study.optimize(func, n_trials=100)

end_time = time.time()

hyper_tuning_time = end_time - start_time

print(f"\tBest value (bcr1p_sum): {study.best_value:.5f}")
print(f"\tBest params:")

for key, value in study.best_params.items():
    print(f"\t\t{key}: {value}")


#### Example  training

In [17]:
best_params = {'n_estimators': 200,
 'learning_rate': 0.008750545843056286,
 'num_leaves': 30,
 'max_depth': 4,
 'min_data_in_leaf': 6,
 'lambda_l1': 0.0002623314818549636,
 'lambda_l2': 1.2647354666343253,
 'bagging_fraction': 0.7521688515598353,
 'bagging_freq': 1,
 'feature_fraction': 0.8860074803941028}

In [22]:

model = lgb.LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
   **best_params,
    n_jobs=-1,
    random_state=42,
    verbose=-1
)
# Train the model
model.fit(
    X_train,
    y_train,
    eval_metric=["binary_logloss", "auc"],
)


LGBMClassifier(bagging_fraction=0.7521688515598353, bagging_freq=1,
               feature_fraction=0.8860074803941028,
               lambda_l1=0.0002623314818549636, lambda_l2=1.2647354666343253,
               learning_rate=0.008750545843056286, max_depth=4,
               min_data_in_leaf=6, n_estimators=200, n_jobs=-1, num_leaves=30,
               objective='binary', random_state=42, verbose=-1)

In [23]:
y_train_prob = model.predict_proba(X_train)[:, 1]
y_test_prob = model.predict_proba(X_test)[:, 1]
y_test_new_prob = model.predict_proba(X_test_new)[:, 1]

thresholds = np.linspace(0.01, 0.99, 100)
f1_scores = [f1_score(y_train, (y_train_prob > t).astype(int)) for t in thresholds]
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

y_train_pred = (y_train_prob > best_threshold).astype(int)
y_test_pred = (y_test_prob > best_threshold).astype(int)
y_test_new_pred = (y_test_new_prob > best_threshold).astype(int)

# Evaluation function
def evaluate(y_true, y_pred, y_prob, label=""):
    print(f"  - Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"  - Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"  - Recall:    {recall_score(y_true, y_pred):.4f}")
    print(f"  - AUC:       {roc_auc_score(y_true, y_prob):.4f}")
    print("")

# Show metrics
print('Train Set: ')
evaluate(y_train, y_train_pred, y_train_prob, label="Train")
print('Test Set: ')
evaluate(y_test, y_test_pred, y_test_prob, label="Test")

Train Set: 
  - Accuracy:  0.9661
  - Precision: 0.9018
  - Recall:    0.8860
  - AUC:       0.9930

Test Set: 
  - Accuracy:  0.5776
  - Precision: 0.8947
  - Recall:    0.1189
  - AUC:       0.7885



#### Setting MLflow Experiment

In [20]:
mlflow.set_experiment("LGBM 2025-11-04 Sydney Siege")

2025/11/09 17:25:16 INFO mlflow.tracking.fluent: Experiment with name 'LGBM 2025-11-04 Sydney Siege' does not exist. Creating a new experiment.


<Experiment: artifact_location='/workspaces/rumour-detection-gnn/New experiments/mlruns/90', creation_time=1762709116811, experiment_id='90', last_update_time=1762709116811, lifecycle_stage='active', name='LGBM 2025-11-04 Sydney Siege', tags={}>

#### Loading dataset statistics to get the final time cut 

In [24]:
df_posts_by_tm = pd.read_csv('sydneysiege_posts_by_time_cut.csv')

df_posts_by_tm['new_posts_cum_sum'] = df_posts_by_tm.new_posts.cumsum()

max_time_cut = int(df_posts_by_tm[df_posts_by_tm.new_posts_cum_sum==int(df_posts_by_tm.new_posts_cum_sum.max())]\
                   .time_cut.min())

In [27]:
import mlflow
import mlflow.lightgbm
import warnings
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, f1_score
import lightgbm as lgb

previous_node_count = 0

for time_cut in range(10, max_time_cut+(60*6), 10):
    print(f"\n=== Time Cut: {time_cut} ===")
    
    processor = Load_Rumours_Dataset_filtering_since_first_post(file_path_replies, file_path_posts, time_cut=time_cut)
    processor.load_data()
    processor.process_data()
    train, test = processor.get_final_dataframes()

    # Prepare features and labels
    X_train  = train.drop(columns=['rumour'])
    X_train = np.hstack([X_train.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_train.embeddings_avg.tolist()))])
    #X = np.hstack([X.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X.embeddings_avg.tolist()))])
    y_train =train['rumour']
    
    X_test  = test.drop(columns=['rumour'])
    X_test_new = test.iloc[previous_node_count:].drop(columns=['rumour'])
    
    X_test_new =  np.hstack([X_test_new.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_test_new.embeddings_avg.tolist()))])
    X_test = np.hstack([X_test.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_test.embeddings_avg.tolist()))])
    
    
    y_test =test['rumour']
    y_test_new = test.iloc[previous_node_count:]['rumour']
    
    previous_node_count = test.shape[0]
    
    print(f"New Instances: {X_test_new.shape[0]}")


    model = lgb.LGBMClassifier(
           objective="binary",
            boosting_type="gbdt",
           **best_params,
            n_jobs=-1,
            random_state=42,
            verbose=-1
    )

    with mlflow.start_run(run_name=f"time_cut_{time_cut}"):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(
                X_train, y_train,
                eval_metric=["binary_logloss", "auc"]
            )

        # Get predicted probabilities
        y_train_prob = model.predict_proba(X_train)[:, 1]
        y_test_prob = model.predict_proba(X_test)[:, 1]
        if X_test_new.shape[0] >0:
            y_test_new_prob = model.predict_proba(X_test_new)[:, 1]

        # Find best threshold maximizing F1 score on training data
        thresholds = np.linspace(0.01, 0.99, 100)
        f1_scores = [f1_score(y_train, (y_train_prob > t).astype(int)) for t in thresholds]
        best_idx = np.argmax(f1_scores)
        best_threshold = thresholds[best_idx]

        # Apply optimal threshold
        y_train_pred = (y_train_prob > best_threshold).astype(int)
        y_test_pred = (y_test_prob > best_threshold).astype(int)
        y_test_new_pred = (y_test_new_prob > best_threshold).astype(int)

        # Log train metrics
        mlflow.log_metric("train_accuracy", accuracy_score(y_train, y_train_pred))
        mlflow.log_metric("train_precision", precision_score(y_train, y_train_pred))
        mlflow.log_metric("train_recall", recall_score(y_train, y_train_pred))
        mlflow.log_metric("train_f1", f1_score(y_train, y_train_pred))
        mlflow.log_metric("train_auc", roc_auc_score(y_train, y_train_prob))

        # Log test metrics
        mlflow.log_metric("final_acc", accuracy_score(y_test, y_test_pred))
        mlflow.log_metric("final_precision", precision_score(y_test, y_test_pred))
        mlflow.log_metric("final_recall", recall_score(y_test, y_test_pred))
        mlflow.log_metric("final_f1", f1_score(y_test, y_test_pred))
        mlflow.log_metric("final_auc", roc_auc_score(y_test, y_test_prob))
        mlflow.log_metric("new_posts", X_test_new.shape[0])

        if X_test_new.shape[0] >0:
            mlflow.log_metric("curr_precision", precision_score(y_test_new, y_test_new_pred))
            mlflow.log_metric("curr_recall", recall_score(y_test_new, y_test_new_pred))
            mlflow.log_metric("curr_acc", accuracy_score(y_test_new, y_test_new_pred))
        else:
            mlflow.log_metric("curr_precision", 0)
            mlflow.log_metric("curr_recall",0)
            mlflow.log_metric("curr_acc", 0)
            

        # Log threshold and time_cut
        mlflow.log_metric("optimal_threshold", best_threshold)
        mlflow.log_metric("time_cut", time_cut)



=== Time Cut: 10 ===
New Instances: 6

=== Time Cut: 20 ===
New Instances: 8

=== Time Cut: 30 ===
New Instances: 8

=== Time Cut: 40 ===
New Instances: 7

=== Time Cut: 50 ===
New Instances: 6

=== Time Cut: 60 ===
New Instances: 6

=== Time Cut: 70 ===
New Instances: 7

=== Time Cut: 80 ===
New Instances: 12

=== Time Cut: 90 ===
New Instances: 9

=== Time Cut: 100 ===
New Instances: 9

=== Time Cut: 110 ===
New Instances: 6

=== Time Cut: 120 ===
New Instances: 8

=== Time Cut: 130 ===
New Instances: 4

=== Time Cut: 140 ===
New Instances: 8


KeyboardInterrupt: 